## weights-track-stats.ipynb
Builds a normalised dense-column sparse matrix of album track statistics
from `sql_feature_album_track_stats.parquet`, aligned to the master
`album_ids.pkl` index.

In [1]:
import pickle
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, save_npz
from sklearn.preprocessing import MinMaxScaler

DATA_DIR     = '../data'
FEATURES_DIR = f'{DATA_DIR}/features'
PARQUET_PATH = f'{DATA_DIR}/sql_feature_album_track_stats.parquet'

In [2]:
# Load master album index (established by 01-album-artist-index.ipynb)
with open(f'{FEATURES_DIR}/album_ids.pkl', 'rb') as f:
    album_ids = pickle.load(f)

album_index = pd.Index(album_ids)
n_albums    = len(album_index)
print(f'Master album universe: {n_albums:,} albums')

Master album universe: 1,008,102 albums


In [3]:
# Columns selected as model features.
# Excluded: pct_tracks_with_length, track_count_with_length (data-quality flags)
#           variance_length_ms (redundant with stddev), range_length_ms (redundant with min/max)
FEATURE_COLS = [
    'first_release_year',
    'medium_count',
    'track_count',
    'total_length_ms',
    'mean_length_ms',
    'median_length_ms',
    'stddev_length_ms',
    'min_length_ms',
    'max_length_ms',
    'p25_length_ms',
    'p75_length_ms',
    'iqr_length_ms',
]

df = pd.read_parquet(PARQUET_PATH, columns=['release_group_id'] + FEATURE_COLS)
df = df.rename(columns={'release_group_id': 'album_id'})

print(f'Raw rows loaded: {len(df):,}')
print(df[FEATURE_COLS].isna().sum().to_string())

Raw rows loaded: 2,235,464
first_release_year    120188
medium_count               0
track_count                0
total_length_ms       213392
mean_length_ms        213392
median_length_ms      213392
stddev_length_ms      213392
min_length_ms         213392
max_length_ms         213392
p25_length_ms         213392
p75_length_ms         213392
iqr_length_ms         213392


In [4]:
# Drop rows where every feature is null (albums with zero tracks in MusicBrainz)
df = df.dropna(subset=FEATURE_COLS, how='all')

# Fill remaining nulls with column medians before scaling
for col in FEATURE_COLS:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

print(f'Albums after null-drop: {len(df):,}')

Albums after null-drop: 2,235,464


In [5]:
# Min-max scale each feature to [0, 1] so no single column dominates by magnitude
scaler = MinMaxScaler()
scaled = scaler.fit_transform(df[FEATURE_COLS].astype(np.float32))

print(f'Scaled array shape: {scaled.shape}')
print(f'Value range: [{scaled.min():.3f}, {scaled.max():.3f}]')

Scaled array shape: (2235464, 12)
Value range: [0.000, 1.000]


In [6]:
# Align to master album index — albums not in the index get zero rows
row_idx = album_index.get_indexer(df['album_id'].values)
valid   = row_idx >= 0

dense_valid = scaled[valid]          # shape (n_valid, n_features)
rows_valid  = row_idx[valid]
n_features  = dense_valid.shape[1]

# Build COO then convert to CSR
col_idx = np.tile(np.arange(n_features), len(rows_valid))
row_rep = np.repeat(rows_valid, n_features)
data    = dense_valid.ravel()

# Drop explicit zeros (scaled values that happen to be 0.0) to stay sparse
nonzero_mask = data != 0.0
X_track_stats = csr_matrix(
    (data[nonzero_mask], (row_rep[nonzero_mask], col_idx[nonzero_mask])),
    shape=(n_albums, n_features)
)

print(f'X_track_stats shape : {X_track_stats.shape}')
print(f'Non-zero entries    : {X_track_stats.nnz:,}')

X_track_stats shape : (1008102, 12)
Non-zero entries    : 11,139,577


In [7]:
save_npz(f'{FEATURES_DIR}/album_track_stats_matrix.npz', X_track_stats)
print('Saved: album_track_stats_matrix.npz')

Saved: album_track_stats_matrix.npz
